In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
con = duckdb.connect("../olist.duckdb", read_only=True)

In [2]:
con.sql("""
SELECT COUNT(*) AS pedidos,
  ROUND(100.0 * AVG((CAST(dt_entrega AS DATE) > CAST(dt_prometida AS DATE))::INT), 2) AS pct_atraso_por_data,
  ROUND(100.0 * AVG((dt_entrega > dt_prometida)::INT), 2)                             AS pct_atraso_por_timestamp
FROM analitica_pedidos
""").df()

,pedidos,pct_atraso_por_data,pct_atraso_por_timestamp
0,96470,6.77,8.11


In [3]:
con.sql("""
SELECT COUNT(*) AS atrasados,
  ROUND(AVG(dias_atraso), 1)         AS media_dias,
  median(dias_atraso)                AS mediana_dias,
  quantile_cont(dias_atraso, 0.9)    AS p90_dias,
  SUM((dias_atraso > 7)::INT)        AS mais_de_7_dias,
  SUM((dias_atraso > 15)::INT)       AS mais_de_15_dias
FROM analitica_pedidos WHERE atrasado
""").df()

,atrasados,media_dias,mediana_dias,p90_dias,mais_de_7_dias,mais_de_15_dias
0,6534,10.6,7.0,22.0,2862.0,1225.0


In [4]:
con.sql("""
SELECT customer_state AS estado, COUNT(*) AS pedidos,
  ROUND(100.0 * AVG(atrasado::INT), 2)  AS pct_atraso,
  ROUND(AVG(prazo_prometido_dias), 1)   AS prazo_prometido_medio,
  ROUND(AVG(prazo_real_dias), 1)        AS prazo_real_medio
FROM analitica_pedidos
GROUP BY 1 ORDER BY pct_atraso DESC
""").df()

,estado,pedidos,pct_atraso,prazo_prometido_medio,prazo_real_medio
0,AL,397,21.41,33.2,24.5
1,MA,717,17.43,31.1,21.5
2,SE,335,15.22,31.5,21.5
3,PI,476,13.87,30.7,19.4
4,CE,1279,13.76,32.0,21.2
5,RR,41,12.20,46.6,29.3
6,BA,3256,12.16,30.1,19.3
7,RJ,12350,12.11,27.0,15.2
8,PA,946,11.21,37.8,23.7
9,ES,1995,10.73,26.2,15.7


In [5]:
con.sql("""
SELECT strftime(dt_compra, '%Y-%m') AS mes, COUNT(*) AS pedidos,
  ROUND(100.0 * AVG(atrasado::INT), 2) AS pct_atraso
FROM analitica_pedidos GROUP BY 1 ORDER BY 1
""").df()

,mes,pedidos,pct_atraso
0,2016-09,1,100.00
1,2016-10,265,0.75
2,2016-12,1,0.00
3,2017-01,750,2.93
4,2017-02,1653,2.96
5,2017-03,2546,4.56
6,2017-04,2303,6.56
7,2017-05,3545,2.99
8,2017-06,3135,3.03
9,2017-07,3872,2.79


In [6]:
con.sql("""
SELECT (customer_state = seller_state) AS mesmo_estado, COUNT(*) AS pedidos,
  ROUND(100.0 * AVG(atrasado::INT), 2) AS pct_atraso,
  ROUND(AVG(prazo_real_dias), 1)       AS prazo_real_medio
FROM analitica_pedidos WHERE seller_state IS NOT NULL
GROUP BY 1
""").df()

,mesmo_estado,pedidos,pct_atraso,prazo_real_medio
0,False,61780,8.05,15.1
1,True,34690,4.51,7.9


In [7]:
con.sql("""
WITH rev AS (
  SELECT order_id, AVG(review_score) AS nota FROM order_reviews GROUP BY order_id
)
SELECT a.atrasado, COUNT(*) AS pedidos,
  ROUND(AVG(r.nota), 2)                     AS nota_media,
  ROUND(100.0 * AVG((r.nota <= 2)::INT), 2) AS pct_nota_baixa
FROM analitica_pedidos a JOIN rev r ON r.order_id = a.order_id
GROUP BY 1
""").df()

,atrasado,pedidos,nota_media,pct_nota_baixa
0,True,6381,2.27,62.36
1,False,89443,4.29,9.23
